In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

In [25]:
# -------------------------
# 1. Data
# -------------------------
# Rule used to label the data:
#   any subject < 35   -> 0 (Fail)
#   else avg <= 60     -> 1 (C)
#   else avg <= 80     -> 2 (B)
#   else               -> 3 (A)
#
# Train and test live in separate files (test is disjoint from train).
# -------------------------

FEATURES = ["subject1", "subject2", "subject3"]

def load(path):
    df = pd.read_csv(path)
    # inputs scaled to 0-1 for stable training (scale predictions the same way)
    X = torch.tensor(df[FEATURES].values, dtype=torch.float32) / 100.0
    y = torch.tensor(df["grade"].values, dtype=torch.long)
    return X, y


X_train, y_train = load("datasets/student_grade_training.csv")
X_test, y_test = load("datasets/student_grade_test.csv")

print("train:", X_train.shape[0], "test:", X_test.shape[0])
print("train class counts:", torch.bincount(y_train).tolist())
print("test  class counts:", torch.bincount(y_test).tolist())

train: 2000 test: 400
train class counts: [500, 500, 500, 500]
test  class counts: [100, 100, 100, 100]


In [26]:
# -------------------------
# 2. Neural Network
# -------------------------

class StudentGradeModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 4),
        )

    def forward(self, x):
        return self.network(x)

model = StudentGradeModel()


In [27]:
# -------------------------
# 3. Loss and Optimizer
# -------------------------

loss_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [28]:
# -------------------------
# 4. Training
# -------------------------

epochs = 10000

for epoch in range(epochs):
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch}, Loss {loss.item():.4f}"
        )
    

Epoch 0, Loss 1.3883
Epoch 100, Loss 1.2607
Epoch 200, Loss 0.9154
Epoch 300, Loss 0.6969
Epoch 400, Loss 0.5517
Epoch 500, Loss 0.4531
Epoch 600, Loss 0.3835
Epoch 700, Loss 0.3317
Epoch 800, Loss 0.2906
Epoch 900, Loss 0.2508
Epoch 1000, Loss 0.2108
Epoch 1100, Loss 0.1838
Epoch 1200, Loss 0.1645
Epoch 1300, Loss 0.1498
Epoch 1400, Loss 0.1378
Epoch 1500, Loss 0.1269
Epoch 1600, Loss 0.1163
Epoch 1700, Loss 0.1077
Epoch 1800, Loss 0.1000
Epoch 1900, Loss 0.0930
Epoch 2000, Loss 0.0871
Epoch 2100, Loss 0.0821
Epoch 2200, Loss 0.0776
Epoch 2300, Loss 0.0736
Epoch 2400, Loss 0.0701
Epoch 2500, Loss 0.0668
Epoch 2600, Loss 0.0637
Epoch 2700, Loss 0.0600
Epoch 2800, Loss 0.0563
Epoch 2900, Loss 0.0531
Epoch 3000, Loss 0.0505
Epoch 3100, Loss 0.0481
Epoch 3200, Loss 0.0458
Epoch 3300, Loss 0.0436
Epoch 3400, Loss 0.0402
Epoch 3500, Loss 0.0380
Epoch 3600, Loss 0.0358
Epoch 3700, Loss 0.0334
Epoch 3800, Loss 0.0317
Epoch 3900, Loss 0.0302
Epoch 4000, Loss 0.0288
Epoch 4100, Loss 0.0276
Epoc

In [30]:
# -------------------------
# 4b. Evaluation on held-out test set
# -------------------------

model.eval()
with torch.no_grad():
    # test loss + overall accuracy
    test_logits = model(X_test)
    test_loss = loss_fn(test_logits, y_test).item()
    test_pred = test_logits.argmax(dim=1)
    acc = (test_pred == y_test).float().mean().item()

    print(f"Test loss:     {test_loss:.4f}")
    print(f"Test accuracy: {acc*100:.2f}%  ({(test_pred == y_test).sum().item()}/{len(y_test)})")

    # per-class accuracy
    print("\nPer-class accuracy:")
    names = {0: "Fail", 1: "C", 2: "B", 3: "A"}
    for c in range(4):
        mask = y_test == c
        c_acc = (test_pred[mask] == y_test[mask]).float().mean().item()
        print(f"  {c} ({names[c]}): {c_acc*100:5.1f}%  (n={mask.sum().item()})")

    # tricky cases: high average but one subject < 35 -> must be Fail
    tricky = torch.tensor(
        [[90, 90, 20],
         [34, 95, 95],
         [35, 35, 35],
         [60, 60, 60],
         [81, 81, 81]],
        dtype=torch.float32
    ) / 100.0
    tricky_pred = model(tricky).argmax(dim=1)
    print("\nTricky cases:")
    for row, p in zip([[90,90,20],[34,95,95],[35,35,35],[60,60,60],[81,81,81]], tricky_pred):
        print(f"  {row} -> {p.item()} ({names[p.item()]})")

Test loss:     0.0503
Test accuracy: 99.00%  (396/400)

Per-class accuracy:
  0 (Fail):  97.0%  (n=100)
  1 (C): 100.0%  (n=100)
  2 (B):  99.0%  (n=100)
  3 (A): 100.0%  (n=100)

Tricky cases:
  [90, 90, 20] -> 0 (Fail)
  [34, 95, 95] -> 0 (Fail)
  [35, 35, 35] -> 1 (C)
  [60, 60, 60] -> 1 (C)
  [81, 81, 81] -> 3 (A)


In [31]:
# -------------------------
# 5. Prediction
# -------------------------

new_student = torch.tensor(
    [[32, 32, 32]],
    dtype=torch.float32
) / 100.0   # scale the same way as training data

model.eval()

with torch.no_grad():
    logits = model(new_student)

    # Convert scores to probabilities
    probabilities = torch.softmax(logits, dim=1)


    predicted_class = torch.argmax(
        probabilities,
        dim=1
    )


print(
    "Predicted class:",
    predicted_class.item()
)


grades = {
    0: "Fail",
    1: "C",
    2: "B",
    3: "A"
}

print(
    "Prediction:",
    grades[predicted_class.item()]
)

Predicted class: 0
Prediction: Fail
